In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Load Data
oil_price = pd.read_csv('../data/processed/Oil/oil_prices_datastream.csv')
ovx = pd.read_csv('../data/processed/Macroeconomic_variables/OVXCLS.csv')

# Ensure datetime format
oil_price['date'] = pd.to_datetime(oil_price['date'], errors='coerce')
ovx['date'] = pd.to_datetime(ovx['date'], errors='coerce')

# 2. Merge data on date (using inner join to ensure we have both)
df = pd.merge(oil_price, ovx, on='date', how='inner').sort_values('date').reset_index(drop=True)

# Use forward fill for any stray NaNs in prices or OVX
df.ffill(inplace=True)

# 3. Calculate Daily Oil Returns (Log returns)
# Assuming the oil price column is named 'price' or similar. Adjust 'oil_price' to your actual column name.
price_col = 'price' if 'price' in df.columns else df.columns[1] # Guessing the column name
df['oil_ret_1d'] = np.log(df[price_col] / df[price_col].shift(1))

df.dropna(inplace=True)

/var/folders/wy/gjw_3_n51t748hfngpz4zf0w0000gn/T/ipykernel_44603/1447220411.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  oil_price['date'] = pd.to_datetime(oil_price['date'], errors='coerce')


In [7]:
# --- PARAMETERS ---
# Define what constitutes a "Jump" (e.g., a daily drop worse than -5%)
JUMP_THRESHOLD = -0.05
TRADING_DAYS_PER_YEAR = 252

# Define OVX Regimes (Bins)
bins = [0, 30, 45, 60, 1000]
labels = ['Low (<30)', 'Medium (30-45)', 'High (45-60)', 'Extreme (>60)']

# Assign each day to an OVX regime based on the OVX close from the PREVIOUS day 
# (to avoid lookahead bias: market sees OVX today, jump happens tomorrow)
df['OVX_Regime'] = pd.cut(df['OVXCLS'].shift(1), bins=bins, labels=labels)

# Boolean flag: 1 if a jump happened today, 0 otherwise
df['is_jump'] = (df['oil_ret_1d'] < JUMP_THRESHOLD).astype(int)

df.dropna(subset=['OVX_Regime'], inplace=True)

df

,date,Brent,WTI,OPEC_basket,Dubai_Crude,OVXCLS,oil_ret_1d,OVX_Regime,is_jump
2,2007-05-14,66.79,62.47,63.79,64.08,27.23,0.008269,Low (<30),0
3,2007-05-15,67.35,63.18,63.79,65.08,27.89,0.008350,Low (<30),0
4,2007-05-16,67.91,62.56,64.55,64.22,27.07,0.008280,Low (<30),0
5,2007-05-17,69.25,64.78,65.29,66.48,24.86,0.019540,Low (<30),0
6,2007-05-18,69.45,64.95,66.01,65.67,24.71,0.002884,Low (<30),0
...,...,...,...,...,...,...,...,...,...
4600,2024-12-26,73.75,70.38,73.92,74.31,30.01,0.000000,Medium (30-45),0
4601,2024-12-27,74.03,71.28,73.91,75.25,30.21,0.003789,Medium (30-45),0
4602,2024-12-30,74.38,71.73,74.62,75.48,30.77,0.004717,Medium (30-45),0
4603,2024-12-31,74.74,72.44,74.59,75.43,30.02,0.004828,Medium (30-45),0


In [8]:
# ============================================================
# FIT CONTINUOUS FUNCTIONS: OVX → jump parameters
# ============================================================
from scipy.optimize import curve_fit

# 1. Prepare data: for each day, we have OVX and whether a jump happened in next 21 days
fit_df = df.dropna(subset=['OVXCLS']).copy()

# --- FIT LAMBDA: logistic(OVX) → P(jump in next 21 days) ---
def logistic(x, a, b):
    return 1 / (1 + np.exp(-(a * x + b)))

# bin OVX into fine grid for fitting (avoids overfitting to individual days)
fit_df['ovx_bin'] = pd.cut(fit_df['OVXCLS'], bins=50)
binned = fit_df.groupby('ovx_bin', observed=True).agg(
    ovx_mid=('OVXCLS', 'mean'),
    prob=('any_jump_next21', 'mean'),
    n=('any_jump_next21', 'count'),
).dropna()
# weight by number of observations per bin
popt_lam, _ = curve_fit(logistic, binned['ovx_mid'], binned['prob'],
                         p0=[0.05, -2], sigma=1/np.sqrt(binned['n']), maxfev=5000)
print(f"Logistic fit for P(jump): a={popt_lam[0]:.4f}, b={popt_lam[1]:.4f}")

# --- FIT MU_J: linear(OVX) → E[worst daily return | jump happened] ---
jumped = fit_df[fit_df['any_jump_next21'] == 1].copy()
jumped['ovx_bin'] = pd.cut(jumped['OVXCLS'], bins=20)
binned_mu = jumped.groupby('ovx_bin', observed=True).agg(
    ovx_mid=('OVXCLS', 'mean'),
    mu=('min_ret_next21', 'mean'),
    sig=('min_ret_next21', 'std'),
    n=('min_ret_next21', 'count'),
).dropna()

# linear fit: mu_J = a + b * OVX
slope_mu, intercept_mu = np.polyfit(binned_mu['ovx_mid'], binned_mu['mu'], 1,
                                     w=np.sqrt(binned_mu['n']))
print(f"Linear fit for mu_J: {intercept_mu:.4f} + {slope_mu:.4f} * OVX")

# linear fit: sigma_J = a + b * OVX
slope_sig, intercept_sig = np.polyfit(binned_mu['ovx_mid'], binned_mu['sig'], 1,
                                       w=np.sqrt(binned_mu['n']))
print(f"Linear fit for sigma_J: {intercept_sig:.4f} + {slope_sig:.4f} * OVX")

# ============================================================
# FINAL FUNCTION
# ============================================================
def ovx_to_jump_params(ovx):
    """Continuous mapping from OVX to jump parameters."""
    if ovx < 30 or np.isnan(ovx):
        return 0.0, 0.0, 0.0
    
    prob = logistic(ovx, *popt_lam)
    lam = -np.log(1 - np.clip(prob, 0, 0.999))
    mu_J = intercept_mu + slope_mu * ovx
    sigma_J = max(intercept_sig + slope_sig * ovx, 0.005)  # floor to prevent 0
    
    return lam, mu_J, sigma_J

# ============================================================
# VISUALIZE THE FITS
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: P(jump) vs OVX
ovx_grid = np.linspace(15, 120, 200)
ax = axes[0]
ax.scatter(binned['ovx_mid'], binned['prob'], s=binned['n']/5, alpha=0.6, label='Empirical')
ax.plot(ovx_grid, logistic(ovx_grid, *popt_lam), 'r-', lw=2, label='Logistic fit')
ax.axvline(30, color='gray', ls='--', alpha=0.5)
ax.set_xlabel('OVX')
ax.set_ylabel('P(jump in next 21 days)')
ax.set_title('Jump Probability')
ax.legend()

# Plot 2: mu_J vs OVX
ax = axes[1]
ax.scatter(binned_mu['ovx_mid'], binned_mu['mu'], s=binned_mu['n']/3, alpha=0.6, label='Empirical')
ax.plot(ovx_grid, intercept_mu + slope_mu * ovx_grid, 'r-', lw=2, label='Linear fit')
ax.axvline(30, color='gray', ls='--', alpha=0.5)
ax.set_xlabel('OVX')
ax.set_ylabel('E[worst daily return]')
ax.set_title('Jump Size (μ_J)')
ax.legend()

# Plot 3: sigma_J vs OVX
ax = axes[2]
ax.scatter(binned_mu['ovx_mid'], binned_mu['sig'], s=binned_mu['n']/3, alpha=0.6, label='Empirical')
ax.plot(ovx_grid, intercept_sig + slope_sig * ovx_grid, 'r-', lw=2, label='Linear fit')
ax.axvline(30, color='gray', ls='--', alpha=0.5)
ax.set_xlabel('OVX')
ax.set_ylabel('Std[worst daily return]')
ax.set_title('Jump Size Volatility (σ_J)')
ax.legend()

fig.suptitle('Continuous OVX → Jump Parameter Mapping', fontsize=13, fontweight='bold')
fig.tight_layout()
plt.show()

# ============================================================
# PRINT PARAMETER TABLE FOR REFERENCE
# ============================================================
print("\nSample parameter values:")
print(f"{'OVX':>6s} {'λ':>8s} {'μ_J':>8s} {'σ_J':>8s}")
for ovx_val in [25, 30, 35, 40, 50, 60, 80, 100]:
    l, m, s = ovx_to_jump_params(ovx_val)
    print(f"{ovx_val:6.0f} {l:8.3f} {m:8.4f} {s:8.4f}")

KeyError: "Label(s) ['any_jump_next21'] do not exist"

In [23]:
# keep the logistic fit for lambda — it's good
# use sample-wide constants for mu_J and sigma_J

mu_J_const = jumped['min_ret_next21'].mean()
sigma_J_const = jumped['min_ret_next21'].std()
print(f"Constant jump size: mu_J = {mu_J_const:.4f}, sigma_J = {sigma_J_const:.4f}")

def ovx_to_jump_params(ovx):
    if ovx < 30 or np.isnan(ovx):
        return 0.0, 0.0, 0.0
    
    prob = logistic(ovx, *popt_lam)
    lam = -np.log(1 - np.clip(prob, 0, 0.999))
    
    return lam, mu_J_const, sigma_J_const

Constant jump size: mu_J = -0.0806, sigma_J = 0.0645


In [24]:
print(f"{'OVX':>6s} {'λ':>8s} {'prob':>8s} {'μ_J':>8s} {'σ_J':>8s}")
for ovx_val in [25, 30, 35, 40, 50, 60, 80, 100]:
    l, m, s = ovx_to_jump_params(ovx_val)
    prob = 1 - np.exp(-l) if l > 0 else 0
    print(f"{ovx_val:6.0f} {l:8.3f} {prob:8.1%} {m:8.4f} {s:8.4f}")

   OVX        λ     prob      μ_J      σ_J
    25    0.000     0.0%   0.0000   0.0000
    30    0.170    15.7%  -0.0806   0.0645
    35    0.242    21.5%  -0.0806   0.0645
    40    0.340    28.8%  -0.0806   0.0645
    50    0.633    46.9%  -0.0806   0.0645
    60    1.074    65.8%  -0.0806   0.0645
    80    2.319    90.2%  -0.0806   0.0645
   100    3.797    97.8%  -0.0806   0.0645
